**LEGACY** — reads a completed `theo_decode_cv.py` output directory (itself
legacy) and has a hardcoded `/gpfs/laur` GLC6962 path. Superseded by
`analyze_param_search.ipynb`, which scans `theo_pipeline_new.py` search
results. Kept for reference.

# Theo pipeline: decoded vs observed, behavior, PSD, frequency diagnostics

Reads a completed `theo_decode_cv.py` output directory and produces, per PC:

1. **`{pc}_traces_by_model.png`** -- one panel per model (black observed vs red
   decoded), then animal-speed and texture-speed panels, then two spectrograms
   of the observed trace at the bottom: raw (absolute dB) and relative (each
   frequency normalized by its own median power across time -- raw dB is
   completely dominated by the low-frequency end, which otherwise makes
   everything else look flat). All panels share one time axis and one set of
   colorbars so they line up exactly.
2. **`{pc}_freq_diagnostics.png`** -- tests whether the decoder captures some
   frequency bands better than others: a PSD overlay (observed vs each
   model's decoded trace) and a coherence spectrum (observed vs each model's
   decoded trace, frequency-resolved).

Figures are saved into `<output-dir>/figures/`, next to the CV run's own figures.

To also get a stationary-gated version of (1): flip `GATE_ON = True` below, set
a real `SPEED_GATE_THRESHOLD`, and re-run -- it saves under a `_gated` suffix
so it doesn't overwrite the ungated set. (2) is not gated -- see the note in
that section.


In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import spectrogram, welch, coherence
from sklearn.metrics import r2_score

# --- Config -------------------------------------------------------------
OUT_DIR = Path(
    "/gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/"
    "theo_pipeline_try4_smooth25_overlap5_nsplits5"
)
FIGS_DIR = OUT_DIR / "figures"
MODELS = ["WF", "WC", "SVR", "XGB"]
MODEL_COLORS = {"WF": "firebrick", "WC": "orangered", "SVR": "crimson", "XGB": "darkred"}

# Stationary gate: when GATE_ON, bins where animal speed >= SPEED_GATE_THRESHOLD are
# blanked (NaN) in every trace panel, and spectrogram columns that are majority-moving
# are blanked too, so gated figures share the same time axis as the ungated ones and
# are directly comparable. Threshold is a placeholder -- tune to whatever "stationary"
# means for this animal/speed units. Only affects section 1; see note in section 2.
GATE_ON = False
SPEED_GATE_THRESHOLD = 0.01

# Time-resolved spectrogram params (section 1): short windows so it can show how
# power changes over the session.
SPEC_NPERSEG = 128
SPEC_NOVERLAP_FRAC = 0.9

# Welch PSD / coherence params (section 2): much longer windows for frequency
# resolution, since this is a single global estimate over the whole session (no
# time axis). fs=5 Hz, nperseg=512 -> ~0.01 Hz resolution, ~50 averaged segments.
COH_NPERSEG = 512
COH_NOVERLAP = 10

GATE_SUFFIX = "_gated" if GATE_ON else ''


In [7]:
# --- Helpers (duplicated from theo_decode_cv.py rather than imported, to avoid
# pulling in its heavier deps -- spikeinterface, matplotlib.use("Agg") side effect,
# etc. -- just to reuse two short functions) ------------------------------------

def to_seconds(series: pd.Series) -> np.ndarray:
    if np.issubdtype(series.dtype, np.timedelta64):
        return series.dt.total_seconds().to_numpy(dtype=float)
    if np.issubdtype(series.dtype, np.datetime64):
        return (series - series.iloc[0]).dt.total_seconds().to_numpy(dtype=float)
    return series.to_numpy(dtype=float)


def bin_output_safe(outputs, output_times, dt, t_start, t_end):
    edges = np.arange(t_start, t_end, dt)
    num_bins = edges.shape[0] - 1
    out = np.full((num_bins, outputs.shape[1]), np.nan, dtype=float)
    for i in range(num_bins):
        m = (output_times >= edges[i]) & (output_times < edges[i + 1])
        if np.any(m):
            out[i, :] = np.nanmean(outputs[m, :], axis=0)
    return out, edges


def compute_spectrograms(y_full, dt, t0, stationary_mask, gate_on):
    """Both a raw (absolute dB) and a relative (each frequency row normalized
    by its own median power across time) spectrogram of the full (continuous)
    trace, padded so both span the exact same time range as the trace panels
    (scipy.signal.spectrogram otherwise drops nperseg/2 samples off each end --
    left unpadded, the heatmap visibly falls short of the trace panels above
    it). Raw dB is dominated by the overall low-frequency-heavy power law --
    everything except the lowest frequencies looks flat -- the relative
    version exists to counter that; both are kept since the raw one is still
    useful as the "ground truth" absolute-power view. When gated, blank any
    column whose underlying window is majority-moving so it renders blank --
    keeps the same time axis as the ungated version.
    """
    fs = 1.0 / dt
    nperseg = min(SPEC_NPERSEG, len(y_full))
    noverlap = int(nperseg * SPEC_NOVERLAP_FRAC)
    step = nperseg - noverlap
    pad = nperseg // 2
    y_padded = np.pad(y_full, pad, mode="reflect")
    freqs, times_spec, Sxx = spectrogram(y_padded, fs=fs, nperseg=nperseg, noverlap=noverlap)

    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    row_median = np.median(Sxx, axis=1, keepdims=True)
    Sxx_rel_db = 10 * np.log10(Sxx / row_median)

    if gate_on:
        moving = ~stationary_mask
        n = len(moving)
        for j in range(Sxx_db.shape[1]):
            start = max(0, j * step - pad)
            stop = min(n, j * step + nperseg - pad)
            if stop > start and moving[start:stop].mean() > 0.5:
                Sxx_db[:, j] = np.nan
                Sxx_rel_db[:, j] = np.nan

    times_abs = t0 + (times_spec - pad / fs)
    raw_vmin, raw_vmax = np.nanpercentile(Sxx_db, [2, 98])
    rel_vlim = np.nanpercentile(np.abs(Sxx_rel_db), 98)
    return (
        freqs, times_abs,
        np.ma.masked_invalid(Sxx_db), (raw_vmin, raw_vmax),
        np.ma.masked_invalid(Sxx_rel_db), rel_vlim,
    )


## Load config, behavior (speed/texture_speed), and model predictions

In [8]:
run_params = json.load(open(OUT_DIR / "run_params.json"))
cv_summary = json.load(open(OUT_DIR / "cv_summary.json"))
cli = run_params["cli_args"]
cfg = cv_summary["config"]

pc_cols = cfg["pc_cols"]
dt = cfg["dt"]
t_start, t_end = cfg["raw_ranges"]["trimmed_overlap"]
n_pc = len(pc_cols)

# --- Behavior: speed + texture_speed, binned onto the exact same dt bins and
# valid_mask (finite across pc_cols) as the CV run, so the time axis lines up
# exactly with predictions/*_oof_predictions.npz's `t`. -------------------------
beh = pd.read_pickle(cli["behavior_df_path"])
if "background" in beh.columns:
    beh = beh[
        (beh["animal"] == cli["animal"])
        & (beh["background"].astype(str).str.lower() == cli["background"].lower())
    ].copy()
else:
    beh = beh[beh["animal"] == cli["animal"]].copy()

beh_t = to_seconds(beh["time"])
beh_outputs = beh[pc_cols + ["speed", "texture_speed"]].to_numpy(dtype=float)
binned, edges = bin_output_safe(beh_outputs, beh_t, dt, t_start, t_end)

valid_mask = np.all(np.isfinite(binned[:, :n_pc]), axis=1)
t_centers = ((edges[:-1] + edges[1:]) / 2.0)[valid_mask]
speed_binned = binned[valid_mask, n_pc]
texture_speed_binned = binned[valid_mask, n_pc + 1]

# --- Model predictions --------------------------------------------------------
preds = {}
for name in MODELS:
    npz = np.load(OUT_DIR / "predictions" / f"{name}_oof_predictions.npz", allow_pickle=True)
    preds[name] = {"t": npz["t"], "y_true": npz["y_true"], "y_pred": npz["y_pred"]}

t_ref = preds[MODELS[0]]["t"]
y_true_ref = preds[MODELS[0]]["y_true"]
for name in MODELS:
    assert np.allclose(preds[name]["t"], t_ref), f"{name}: t mismatch across models"
    assert np.allclose(preds[name]["y_true"], y_true_ref, equal_nan=True), f"{name}: y_true mismatch across models"
assert np.allclose(t_ref, t_centers), "Recomputed behavior time axis does not match predictions' t"
print(f"OK: {len(t_centers)} time bins, {n_pc} PCs, {len(MODELS)} models, all aligned.")

stationary_mask = speed_binned < SPEED_GATE_THRESHOLD if GATE_ON else np.ones_like(speed_binned, dtype=bool)
print(f"GATE_ON={GATE_ON}" + (f", stationary fraction={stationary_mask.mean():.2%}" if GATE_ON else ""))


def gated(arr):
    return np.where(stationary_mask, arr, np.nan) if GATE_ON else arr


OK: 13256 time bins, 5 PCs, 4 models, all aligned.
GATE_ON=False


## Section 1: per-model decoded-vs-observed traces, speed, texture-speed, PSD

In [ ]:
FIGS_DIR.mkdir(parents=True, exist_ok=True)

for pc_idx, pc in enumerate(pc_cols):
    y_true_pc = y_true_ref[:, pc_idx]
    freqs, times_abs, Sxx_db_masked, (raw_vmin, raw_vmax), Sxx_rel_masked, rel_vlim = compute_spectrograms(
        y_true_pc, dt, t_centers[0], stationary_mask, GATE_ON
    )

    n_panels = len(MODELS) + 4
    fig, axes = plt.subplots(
        n_panels, 1, figsize=(16, 2.2 * len(MODELS) + 2 * 1.6 + 2 * 2.2), sharex=True,
        gridspec_kw={"height_ratios": [1] * len(MODELS) + [0.8, 0.8, 2, 2]},
        constrained_layout=True,
    )

    for i, name in enumerate(MODELS):
        ax = axes[i]
        yt = gated(y_true_pc)
        yp = gated(preds[name]["y_pred"][:, pc_idx])
        valid = np.isfinite(yt) & np.isfinite(yp)
        r2 = r2_score(y_true_pc[valid], preds[name]["y_pred"][valid, pc_idx]) if valid.any() else float("nan")
        ax.plot(t_centers, yt, color="black", lw=0.7, label="Observed")
        ax.plot(t_centers, yp, color="tab:red", lw=0.7, alpha=0.85, label="Decoded")
        ax.set_ylabel(f"{name}\n(R2={r2:.2f})")
        ax.grid(alpha=0.2)
        if i == 0:
            ax.legend(loc="upper right", fontsize=8)

    ax = axes[len(MODELS)]
    ax.plot(t_centers, gated(speed_binned), color="tab:blue", lw=0.7)
    if GATE_ON:
        ax.axhline(SPEED_GATE_THRESHOLD, color="gray", lw=0.8, ls="--")
    ax.set_ylabel("Animal\nspeed")
    ax.grid(alpha=0.2)

    ax = axes[len(MODELS) + 1]
    ax.plot(t_centers, gated(texture_speed_binned), color="tab:purple", lw=0.7)
    ax.set_ylabel("Texture\nspeed")
    ax.grid(alpha=0.2)

    ax_raw = axes[-2]
    pcm_raw = ax_raw.pcolormesh(
        times_abs, freqs, Sxx_db_masked, shading="gouraud", cmap="viridis", vmin=raw_vmin, vmax=raw_vmax
    )
    ax_raw.set_ylabel("Freq (Hz)\n(raw)")

    ax_rel = axes[-1]
    pcm_rel = ax_rel.pcolormesh(
        times_abs, freqs, Sxx_rel_masked, shading="gouraud", cmap="RdBu_r", vmin=-rel_vlim, vmax=rel_vlim
    )
    ax_rel.set_ylabel("Freq (Hz)\n(relative)")
    ax_rel.set_xlabel("Time (s)")

    # Attach each colorbar to every axes in the figure (not just its own spectrogram's),
    # so every row's plot area shrinks by the same amount -- otherwise only the
    # spectrogram row(s) narrow to make room for their colorbar, and visibly stop
    # short of the trace panels' time axis above them even though the data (and
    # xlim) are the same.
    fig.colorbar(pcm_raw, ax=list(axes), label="Power (dB)", pad=0.02, fraction=0.03, aspect=40)
    fig.colorbar(
        pcm_rel, ax=list(axes), label="Relative power (dB vs. that freq's median)",
        pad=0.02, fraction=0.03, aspect=40,
    )

    title = f"{pc}: decoded vs observed, behavior, PSD"
    if GATE_ON:
        title += f" (gated: speed < {SPEED_GATE_THRESHOLD})"
    fig.suptitle(title)

    out_path = FIGS_DIR / f"{pc}_traces_by_model{GATE_SUFFIX}.png"
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print("Saved", out_path)


## Section 2: frequency diagnostics -- PSD overlay + coherence

Tests whether the decoder tracks some frequency bands better than others, directly
(rather than eyeballing the spectrogram).

**How coherence is calculated** (`scipy.signal.coherence`, magnitude-squared
coherence): split observed (x) and decoded (y) into overlapping windows
(`COH_NPERSEG` long, `COH_NOVERLAP` overlap -- same windowing Welch's method
uses for a PSD), Fourier-transform each window, then for each frequency f:

```
Cxy(f) = |Pxy(f)|^2 / (Pxx(f) * Pyy(f))
```

`Pxx`/`Pyy` are each signal's own average power at f across all windows;
`Pxy` is the *complex* cross-power between x and y at f, averaged across
windows -- complex because it carries both amplitude ratio and phase (timing)
offset per window, not just magnitude. That averaging is what makes this
meaningful: if x and y are consistently related at frequency f (fixed
amplitude ratio and phase lag, window after window), their complex products
reinforce and `|Pxy|` stays large relative to `Pxx*Pyy`, giving coherence near
1. If the relationship at f is inconsistent across windows -- even if each
individual window looks correlated -- the complex products point in random
directions and mostly cancel when averaged, giving coherence near 0. This is
why coherence needs multiple averaged windows to mean anything: with only one
window (no averaging), it is trivially 1.0 everywhere and tells you nothing.
Coherence is effectively a frequency-resolved R2: 0 = unrelated at that
frequency, 1 = fully, consistently linearly related (right amplitude ratio
*and* right timing) at that frequency.

The PSD overlay is a simpler complement -- it shows whether the decoded trace
has power at a given frequency at all, but not whether it's correctly *timed*
there (unlike coherence).

Not gated: Welch/coherence need a single contiguous, uniformly-sampled series;
honoring `GATE_ON` here would mean either splicing together non-adjacent
stationary runs (fabricating false continuity at the splice points) or a
per-run segmented Welch variant -- more machinery than built here. If you want
this restricted to stationary epochs, ask and I'll add the run-respecting
version.

**Resolution caveat**: `COH_NPERSEG=512` (~102 s window) resolves down to
~0.01 Hz (~100 s period) with ~50 averaged windows (a stable estimate). It
cannot say anything about genuinely slower, session-scale trends (the
multi-hundred-second drifts visible by eye in section 1) -- resolving those
with any statistical confidence would need many repeats of that timescale
within the session, i.e. a session many times longer than this one. Treat any
conclusion here as scoped to ~0.01-2.5 Hz, not as a verdict on the slowest
trends.


In [10]:
n_samples = len(t_centers)
fs = 1.0 / dt
step = COH_NPERSEG - COH_NOVERLAP
L = 1 + (n_samples - COH_NPERSEG) // step
print(
    f"Welch/coherence: nperseg={COH_NPERSEG} ({COH_NPERSEG / fs:.1f}s window), "
    f"{L} averaged windows, frequency floor ~{fs / COH_NPERSEG:.4f} Hz"
)

for pc_idx, pc in enumerate(pc_cols):
    y_true_pc = y_true_ref[:, pc_idx]

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    f_psd, Pxx_obs = welch(y_true_pc, fs=fs, nperseg=COH_NPERSEG, noverlap=COH_NOVERLAP)
    axes[0].semilogy(f_psd[1:], Pxx_obs[1:], color="black", lw=1.5, label="Observed")
    for name in MODELS:
        f_psd, Pxx_pred = welch(preds[name]["y_pred"][:, pc_idx], fs=fs, nperseg=COH_NPERSEG, noverlap=COH_NOVERLAP)
        axes[0].semilogy(f_psd[1:], Pxx_pred[1:], color=MODEL_COLORS[name], lw=1.1, alpha=0.9, label=name)
    axes[0].set_xscale("log")
    axes[0].set_ylabel("PSD (power/Hz)")
    axes[0].set_title(f"{pc}: PSD -- observed vs decoded")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.2, which="both")

    for name in MODELS:
        f_coh, Cxy = coherence(
            y_true_pc, preds[name]["y_pred"][:, pc_idx], fs=fs, nperseg=COH_NPERSEG, noverlap=COH_NOVERLAP
        )
        axes[1].plot(f_coh[1:], Cxy[1:], color=MODEL_COLORS[name], lw=1.3, alpha=0.9, label=name)
    axes[1].set_xscale("log")
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Coherence")
    axes[1].set_title(f"{pc}: coherence (observed vs decoded)")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.2, which="both")

    fig.tight_layout()
    out_path = FIGS_DIR / f"{pc}_freq_diagnostics.png"
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print("Saved", out_path)


Welch/coherence: nperseg=512 (102.4s window), 26 averaged windows, frequency floor ~0.0098 Hz
Saved /gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/theo_pipeline_try4_smooth25_overlap5_nsplits5/figures/PC1_freq_diagnostics.png
Saved /gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/theo_pipeline_try4_smooth25_overlap5_nsplits5/figures/PC2_freq_diagnostics.png
Saved /gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/theo_pipeline_try4_smooth25_overlap5_nsplits5/figures/PC3_freq_diagnostics.png
Saved /gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/theo_pipeline_try4_smooth25_overlap5_nsplits5/figures/PC4_freq_diagnostics.png
Saved /gpfs/laur/data/0_collaboration/ceph_phys/GLC6962/GLC6962_g_g0/theo_pipeline_try4_smooth25_overlap5_nsplits5/figures/PC5_freq_diagnostics.png


## Section 3: spectrograms -- observed, decoded, and residual, per model

For each PC: top row is the observed trace's own spectrogram (raw + relative
only, aligned under the first two columns -- it has no decoded/residual
columns). Below that, one row per model, four columns: decoded raw (dB),
decoded relative to its own per-frequency median (dB), residual raw (dB),
residual relative (dB). Same colormap convention as `{pc}_traces_by_model.png`:
raw absolute-power dB (`viridis`), relative power (`RdBu_r`, ratio to that
row's own per-frequency median power -- see the markdown/code for the exact
`10*log10(Sxx / median(Sxx, axis=time))` formula). Uses the same
`compute_spectrograms` helper and `SPEC_NPERSEG`/`SPEC_NOVERLAP_FRAC` params as
section 1.

Color limits are shared across observed, decoded, and residual, per column
type (raw vs. relative), so brightness is directly comparable throughout the
whole figure -- e.g. whether a residual still carries substantial power in a
given time/frequency region, relative to how much the observed trace had
there. Both colorbars are placed to the right of the whole grid.


In [ ]:
for pc_idx, pc in enumerate(pc_cols):
    y_true_pc = y_true_ref[:, pc_idx]
    r2_by_model = {name: r2_score(y_true_pc, preds[name]["y_pred"][:, pc_idx]) for name in MODELS}

    freqs, times_abs, obs_db, _, obs_rel, _ = compute_spectrograms(
        y_true_pc, dt, t_centers[0], stationary_mask, GATE_ON
    )

    rows = []
    for name in MODELS:
        decoded = preds[name]["y_pred"][:, pc_idx]
        residual = y_true_pc - decoded
        _, _, dec_db, _, dec_rel, _ = compute_spectrograms(
            decoded, dt, t_centers[0], stationary_mask, GATE_ON
        )
        _, _, res_db, _, res_rel, _ = compute_spectrograms(
            residual, dt, t_centers[0], stationary_mask, GATE_ON
        )
        rows.append((name, dec_db, dec_rel, res_db, res_rel))

    # Shared color scale across observed, decoded, and residual, per type (raw vs
    # relative), so brightness is directly comparable throughout the whole figure.
    raw_all = np.concatenate(
        [np.ma.compressed(obs_db)]
        + [np.ma.compressed(r[1]) for r in rows]
        + [np.ma.compressed(r[3]) for r in rows]
    )
    rel_all = np.concatenate(
        [np.ma.compressed(obs_rel)]
        + [np.ma.compressed(r[2]) for r in rows]
        + [np.ma.compressed(r[4]) for r in rows]
    )
    raw_vmin, raw_vmax = np.percentile(raw_all, [2, 98])
    rel_vlim = np.percentile(np.abs(rel_all), 98)

    col_titles = ["Decoded raw (dB)", "Decoded relative (dB)", "Residual raw (dB)", "Residual relative (dB)"]
    n_rows = len(MODELS) + 1
    fig, axes = plt.subplots(
        n_rows, 4, figsize=(20, 2.0 * n_rows), sharex=True, sharey=True,
        constrained_layout=True,
    )

    # Row 0: observed, raw + relative only -- no decoded/residual columns apply to it.
    pcm_raw = axes[0, 0].pcolormesh(
        times_abs, freqs, obs_db, shading="gouraud", cmap="viridis", vmin=raw_vmin, vmax=raw_vmax
    )
    axes[0, 0].set_title("Observed raw (dB)")
    axes[0, 0].set_ylabel("Observed\nFreq (Hz)")
    pcm_rel = axes[0, 1].pcolormesh(
        times_abs, freqs, obs_rel, shading="gouraud", cmap="RdBu_r", vmin=-rel_vlim, vmax=rel_vlim
    )
    axes[0, 1].set_title("Observed relative (dB)")
    axes[0, 2].set_visible(False)
    axes[0, 3].set_visible(False)

    for i, (name, dec_db, dec_rel, res_db, res_rel) in enumerate(rows, start=1):
        panels = [
            (dec_db, "viridis", raw_vmin, raw_vmax),
            (dec_rel, "RdBu_r", -rel_vlim, rel_vlim),
            (res_db, "viridis", raw_vmin, raw_vmax),
            (res_rel, "RdBu_r", -rel_vlim, rel_vlim),
        ]
        for j, (data, cmap, vmin, vmax) in enumerate(panels):
            ax = axes[i, j]
            ax.pcolormesh(times_abs, freqs, data, shading="gouraud", cmap=cmap, vmin=vmin, vmax=vmax)
            if i == 1:
                ax.set_title(col_titles[j])
        axes[i, 0].set_ylabel(f"{name} (R2={r2_by_model[name]:.2f})\nFreq (Hz)")

    for j in range(4):
        axes[-1, j].set_xlabel("Time (s)")

    # Both colorbars go on the right of the whole grid (not wedged between columns) --
    # attach to every axes, matching the pattern already used in section 1.
    fig.colorbar(pcm_raw, ax=list(axes.ravel()), label="Power (dB)", pad=0.02, fraction=0.03, aspect=40)
    fig.colorbar(
        pcm_rel, ax=list(axes.ravel()), label="Relative power (dB vs. that freq's median)",
        pad=0.02, fraction=0.03, aspect=40,
    )

    fig.suptitle(f"{pc}: observed, decoded, and residual spectrograms, per model")

    out_path = FIGS_DIR / f"{pc}_spectrograms_by_trace.png"
    fig.savefig(out_path, dpi=130)
    plt.close(fig)
    print("Saved", out_path)
